# drafter-lora

Does a small LoRA adapter on an EAGLE-3 draft head restore speculative decoding speed after the
target model is LoRA fine-tuned?

**Before running:** Settings → Accelerator → **GPU T4 x2** (only one GPU is used), and
Settings → **Internet on** (needs a phone-verified Kaggle account).

Run one stage at a time. Each stage runs a small smoke test first, then the real run.
At the end of each stage, copy the printed tables back to Claude and download
`/kaggle/working/drafter-lora-results.zip` (Kaggle deletes files when the session ends).

## Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WORK"] = "/kaggle/working"
os.environ["PYTHONPATH"] = "/kaggle/working/EAGLE:/kaggle/working/drafter-lora"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "transformers==4.53.3" "peft==0.16.0"
!test -d /kaggle/working/EAGLE || git clone -q https://github.com/SafeAILab/EAGLE.git /kaggle/working/EAGLE
!cd /kaggle/working/EAGLE && git checkout -q cb7e0841fe0c206c6ed74a197ad5e2a1f13f5a2b && git log -1 --format="EAGLE commit %h (%cd)"
!python -c "import torch, transformers, peft; print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '|', torch.cuda.get_device_name(0))"
os.makedirs("/kaggle/working/drafter-lora", exist_ok=True)
%cd /kaggle/working/drafter-lora

In [ ]:
# When earlier stages ran as separate Kaggle notebooks attached as inputs, bring their outputs in.
import glob
for src in glob.glob("/kaggle/input/**/drafter-lora", recursive=True):
    for sub in ["results", "prompts", "data", "adapters"]:
        if os.path.isdir(f"{src}/{sub}"):
            !cp -rn {src}/{sub} .
for found in glob.glob("/kaggle/input/**/target_lora*", recursive=True):  # saved target LoRA adapters
    name = os.path.basename(found)
    if os.path.isdir(found) and not os.path.exists(f"/kaggle/working/{name}"):
        os.symlink(found, f"/kaggle/working/{name}")
!ls results prompts 2>/dev/null; ls -l /kaggle/working | grep target || true

## Scripts
Each cell writes one script into `/kaggle/working/drafter-lora`.

In [ ]:
%%writefile common.py
"""Shared settings and helpers for the drafter-lora scripts."""
import json
import os
import random

import numpy as np
import torch

BASE = "Qwen/Qwen3-1.7B"
HEAD = "AngelSlim/Qwen3-1.7B_eagle3"
WORK = os.environ.get("WORK", "/kaggle/working")
SEED = 42

# EAGLE-3 drafting settings, fixed for every run so results are comparable.
TOTAL_TOKEN = 60
DEPTH = 7
TOP_K = 10


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def chat_ids(tokenizer, prompt):
    """Token ids for a single user turn, ready for the assistant to answer (thinking off)."""
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    return tokenizer(text, add_special_tokens=False).input_ids


def medmcqa_prompt(ex):
    opts = "\n".join(f"{l}. {ex['op' + l.lower()]}" for l in "ABCD")
    return f"{ex['question']}\n\n{opts}\n\nAnswer with the correct option and a short explanation."


def medmcqa_answer(ex):
    l = "ABCD"[ex["cop"]]
    return f"Answer: {l}. {ex['op' + l.lower()]}\n\nExplanation: {ex['exp'].strip()}"


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(path, rows):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")


def load_ea_model(base_path, head_path=HEAD):
    from eagle.model.ea_model import EaModel

    model = EaModel.from_pretrained(
        base_model_path=base_path,
        ea_model_path=head_path,
        use_eagle3=True,
        total_token=TOTAL_TOKEN,
        depth=DEPTH,
        top_k=TOP_K,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map={"": 0},
    )
    model.eval()
    return model

In [ ]:
%%writefile head_lora.py
"""Add, save and load a LoRA adapter on the EAGLE-3 draft head (model.ea_layer)."""
import json
import os

import torch
from peft import LoraConfig, inject_adapter_in_model
from peft.tuners.lora import LoraLayer

# Every linear layer in the head except lm_head: the attention + MLP of the single
# decoder layer, and fc (the projection of the target's 3 hidden-state taps).
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "fc"]


def add_head_lora(head, rank):
    cfg = LoraConfig(r=rank, lora_alpha=2 * rank, lora_dropout=0.0, target_modules=TARGET_MODULES)
    inject_adapter_in_model(cfg, head)
    for name, param in head.named_parameters():
        param.requires_grad = "lora_" in name
    return {"rank": rank, "alpha": 2 * rank, "target_modules": TARGET_MODULES}


def save_head_adapter(head, meta, folder):
    os.makedirs(folder, exist_ok=True)
    sd = {n: p.detach().to(torch.float16).cpu() for n, p in head.named_parameters() if "lora_" in n}
    torch.save(sd, os.path.join(folder, "head_lora.pt"))
    with open(os.path.join(folder, "adapter_config.json"), "w") as f:
        json.dump(meta, f, indent=2)
    return sd


def apply_head_adapter(head, folder):
    """Load the adapter, merge it into the head's weights, and return the head in fp16."""
    meta = json.load(open(os.path.join(folder, "adapter_config.json")))
    cfg = LoraConfig(r=meta["rank"], lora_alpha=meta["alpha"], lora_dropout=0.0,
                     target_modules=meta["target_modules"])
    inject_adapter_in_model(cfg, head)
    sd = torch.load(os.path.join(folder, "head_lora.pt"), map_location="cpu")
    lora_keys = {n for n, _ in head.named_parameters() if "lora_" in n}
    if set(sd) != lora_keys:
        raise ValueError(f"adapter keys do not match the head: {sorted(set(sd) ^ lora_keys)[:5]}")
    head.load_state_dict(sd, strict=False)
    for m in head.modules():
        if isinstance(m, LoraLayer):
            m.merge()
    head.to(torch.float16)
    print(f"merged head adapter from {folder} ({sum(v.numel() for v in sd.values()):,} params)")

In [ ]:
%%writefile make_prompts.py
"""Build the fixed prompt sets used by every stage.

prompts/general_eval.jsonl   100 Alpaca instructions (Stage 1 baseline)
prompts/med_eval.jsonl       100 MedMCQA validation questions (Stage 2/3 eval)
data/med_train.jsonl         MedMCQA train questions + answers (Stage 2 target LoRA)
prompts/med_headtrain.jsonl  MedMCQA train questions, disjoint from the above (Stage 3 head data)
"""
import argparse
import os
import re

from datasets import load_dataset

from common import SEED, medmcqa_answer, medmcqa_prompt, write_jsonl

p = argparse.ArgumentParser()
p.add_argument("--out", default=".")
p.add_argument("--n_eval", type=int, default=100)
p.add_argument("--n_target_train", type=int, default=1000)
p.add_argument("--n_head_train", type=int, default=600)
args = p.parse_args()

alpaca = load_dataset("tatsu-lab/alpaca", split="train").shuffle(seed=SEED)
alpaca = alpaca.filter(lambda ex: ex["input"] == "").select(range(args.n_eval))
write_jsonl(os.path.join(args.out, "prompts/general_eval.jsonl"),
            [{"id": f"alpaca-{i}", "prompt": ex["instruction"]} for i, ex in enumerate(alpaca)])


def usable(ex):
    return ex["exp"] is not None and 50 <= len(ex["exp"]) <= 1500 and ex["choice_type"] == "single"


def clean_for_training(ex):
    """Training answers only: drop explanations with answer-key prefixes ("Ans. is 'a' i.e. ..."),
    textbook citations or run-together sentences. The first LoRA trained on these looped badly."""
    if not usable(ex):
        return False
    e = ex["exp"].strip()
    return (100 <= len(e) <= 800
            and not re.match(r"(?i)^\s*ans", e)
            and not re.search(r"(?i)\bref\b|reference|textbook|\bpg\b|edition", e)
            and not re.search(r"[a-z]\.[A-Z]", e))


val = load_dataset("openlifescienceai/medmcqa", split="validation").shuffle(seed=SEED).filter(usable)
write_jsonl(os.path.join(args.out, "prompts/med_eval.jsonl"),
            [{"id": ex["id"], "prompt": medmcqa_prompt(ex)} for ex in val.select(range(args.n_eval))])

train = load_dataset("openlifescienceai/medmcqa", split="train").shuffle(seed=SEED).filter(clean_for_training)
n1, n2 = args.n_target_train, args.n_target_train + args.n_head_train
write_jsonl(os.path.join(args.out, "data/med_train.jsonl"),
            [{"id": ex["id"], "prompt": medmcqa_prompt(ex), "answer": medmcqa_answer(ex)}
             for ex in train.select(range(n1))])
write_jsonl(os.path.join(args.out, "prompts/med_headtrain.jsonl"),
            [{"id": ex["id"], "prompt": medmcqa_prompt(ex)} for ex in train.select(range(n1, n2))])

print(f"general_eval={len(alpaca)} med_eval={args.n_eval} med_train={n1} med_headtrain={n2 - n1}")

In [ ]:
%%writefile check_head.py
"""Stage 1 smoke test: does the EAGLE-3 head load cleanly and produce the same text as plain decoding?"""
import argparse

import torch
from huggingface_hub import hf_hub_download

from common import BASE, HEAD, chat_ids, load_ea_model, set_seed

p = argparse.ArgumentParser()
p.add_argument("--base", default=BASE)
p.add_argument("--head", default=HEAD)
args = p.parse_args()
set_seed()

model = load_ea_model(args.base, args.head)
tok = model.get_tokenizer()

# EaModel loads the head with strict=False, so check the keys ourselves.
sd = torch.load(hf_hub_download(args.head, "pytorch_model.bin"), map_location="cpu")
expected = set(model.ea_layer.state_dict()) - {"embed_tokens.weight"}  # embeddings come from the target
missing = sorted(expected - set(sd))
unexpected = sorted(set(sd) - expected)
print("head tensors in file:", len(sd))
print("missing keys:", missing or "none")
print("unexpected keys:", unexpected or "none")

prompt = "Explain in three sentences why the sky is blue."
ids = torch.tensor([chat_ids(tok, prompt)], device="cuda")
out_ea, new_ea, idx = model.eagenerate(ids, temperature=0.0, max_new_tokens=128, log=True)
out_base, new_base, _ = model.naivegenerate(ids, temperature=0.0, max_new_tokens=128, log=True)
text_ea = tok.decode(out_ea[0, ids.shape[1]:], skip_special_tokens=True)
text_base = tok.decode(out_base[0, ids.shape[1]:], skip_special_tokens=True)
print("\n--- EAGLE output ---\n" + text_ea)
print("\n--- plain output ---\n" + text_base)
print(f"\naccepted tokens/step on this prompt: {int(new_ea) / (int(idx) + 1):.2f}")
print("outputs identical:", text_ea == text_base)
if missing or unexpected:
    raise SystemExit("Head keys do not match the model. Stop and investigate before benchmarking.")

In [ ]:
%%writefile bench.py
"""Benchmark speculative decoding vs plain decoding on a prompt set; one CSV row per prompt.

accept_len = new tokens / draft-verify steps (includes the target's bonus token, as in the EAGLE papers).
Resumable: prompts already in --out are skipped.
"""
import argparse
import csv
import os
import time

import torch

from common import BASE, HEAD, chat_ids, load_ea_model, read_jsonl, set_seed
from head_lora import apply_head_adapter

p = argparse.ArgumentParser()
p.add_argument("--base", default=BASE, help="target model (HF id or merged LoRA folder)")
p.add_argument("--head", default=HEAD)
p.add_argument("--adapter", default=None, help="folder with a draft-head LoRA adapter")
p.add_argument("--prompts", required=True)
p.add_argument("--out", required=True)
p.add_argument("--n", type=int, default=100)
p.add_argument("--max_new_tokens", type=int, default=256)
args = p.parse_args()
set_seed()

model = load_ea_model(args.base, args.head)
if args.adapter:
    apply_head_adapter(model.ea_layer, args.adapter)
tok = model.get_tokenizer()


def timed(fn, ids):
    torch.cuda.synchronize()
    t = time.perf_counter()
    out, new_tokens, idx = fn(ids, temperature=0.0, max_new_tokens=args.max_new_tokens, log=True)
    torch.cuda.synchronize()
    return out[0, ids.shape[1]:].tolist(), int(new_tokens), int(idx) + 1, time.perf_counter() - t


prompts = read_jsonl(args.prompts)[: args.n]
warm = torch.tensor([chat_ids(tok, prompts[0]["prompt"])], device="cuda")
timed(model.eagenerate, warm)
timed(model.naivegenerate, warm)

fields = ["id", "input_len", "ea_new_tokens", "ea_steps", "accept_len", "ea_time_s", "ea_tok_per_s",
          "base_new_tokens", "base_time_s", "base_tok_per_s", "speedup", "same_output", "ea_text"]
done = set()
if os.path.exists(args.out):
    done = {r["id"] for r in csv.DictReader(open(args.out))}
os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
f = open(args.out, "a", newline="")
w = csv.DictWriter(f, fieldnames=fields)
if not done:
    w.writeheader()

for i, ex in enumerate(prompts):
    if ex["id"] in done:
        continue
    ids = torch.tensor([chat_ids(tok, ex["prompt"])], device="cuda")
    ea_out, ea_new, ea_steps, ea_t = timed(model.eagenerate, ids)
    base_out, base_new, _, base_t = timed(model.naivegenerate, ids)
    n = min(len(ea_out), len(base_out))
    row = {
        "id": ex["id"], "input_len": ids.shape[1],
        "ea_new_tokens": ea_new, "ea_steps": ea_steps, "accept_len": round(ea_new / ea_steps, 4),
        "ea_time_s": round(ea_t, 4), "ea_tok_per_s": round(ea_new / ea_t, 2),
        "base_new_tokens": base_new, "base_time_s": round(base_t, 4), "base_tok_per_s": round(base_new / base_t, 2),
        "speedup": round((ea_new / ea_t) / (base_new / base_t), 4),
        "same_output": ea_out[:n] == base_out[:n],
        "ea_text": tok.decode(ea_out, skip_special_tokens=True),
    }
    w.writerow(row)
    f.flush()
    print(f"[{i + 1}/{len(prompts)}] accept_len={row['accept_len']:.2f} "
          f"ea={row['ea_tok_per_s']:.1f} tok/s base={row['base_tok_per_s']:.1f} tok/s")
f.close()

In [ ]:
%%writefile summarize.py
"""Print and save a summary table from bench CSVs.

Usage: python summarize.py --out results/summary.csv "baseline (general)=results/stage1_baseline.csv" ...
"""
import argparse
import csv
import os


def looping(text, n=8, threshold=0.3):
    """True if at least 30% of the output's 8-word chunks are repeats (the model is stuck in a loop)."""
    w = text.split()
    grams = [" ".join(w[i:i + n]) for i in range(len(w) - n + 1)]
    return bool(grams) and 1 - len(set(grams)) / len(grams) >= threshold


def main():
    p = argparse.ArgumentParser()
    p.add_argument("runs", nargs="+", help="label=path.csv")
    p.add_argument("--out", default=None)
    args = p.parse_args()

    rows = []
    for run in args.runs:
        label, path = run.split("=", 1)
        data = list(csv.DictReader(open(path)))
        num = lambda k: [float(r[k]) for r in data]
        ea_tok, ea_t = sum(num("ea_new_tokens")), sum(num("ea_time_s"))
        base_tok, base_t = sum(num("base_new_tokens")), sum(num("base_time_s"))
        ok = [r for r in data if not looping(r["ea_text"])]
        rows.append({
            "run": label,
            "prompts": len(data),
            "accept_len": round(sum(num("ea_new_tokens")) / sum(num("ea_steps")), 3),
            "ea_tok_per_s": round(ea_tok / ea_t, 2),
            "base_tok_per_s": round(base_tok / base_t, 2),
            "speedup": round((ea_tok / ea_t) / (base_tok / base_t), 3),
            "looping_pct": round(100 * (len(data) - len(ok)) / len(data), 1),
            "accept_len_no_loops": round(sum(float(r["ea_new_tokens"]) for r in ok) / sum(float(r["ea_steps"]) for r in ok), 3) if ok else "",
            "same_output_pct": round(100 * sum(r["same_output"] == "True" for r in data) / len(data), 1),
        })

    cols = list(rows[0])
    widths = [max(len(c), *(len(str(r[c])) for r in rows)) for c in cols]
    print("  ".join(c.ljust(w) for c, w in zip(cols, widths)))
    for r in rows:
        print("  ".join(str(r[c]).ljust(w) for c, w in zip(cols, widths)))

    if args.out:
        os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
        with open(args.out, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            w.writerows(rows)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile paired.py
"""Per-prompt paired comparison of accepted tokens/step between two runs on the same prompts.

Prompts where either run loops (see summarize.looping) are excluded. The 95% interval is a
bootstrap over prompts (4,000 resamples, seed 0).

Usage: python paired.py --out results/paired_stats.csv "name=before.csv,after.csv" ...
"""
import argparse
import csv
import os
import random
import statistics as st

from summarize import looping

p = argparse.ArgumentParser()
p.add_argument("pairs", nargs="+", help="label=before.csv,after.csv")
p.add_argument("--out", required=True)
args = p.parse_args()

rows = []
for pair in args.pairs:
    label, paths = pair.split("=", 1)
    before, after = ({r["id"]: r for r in csv.DictReader(open(path))} for path in paths.split(","))
    ids = [k for k in before if not looping(before[k]["ea_text"]) and not looping(after[k]["ea_text"])]
    diffs = [float(after[k]["accept_len"]) - float(before[k]["accept_len"]) for k in ids]
    random.seed(0)
    boot = sorted(st.mean(random.choices(diffs, k=len(diffs))) for _ in range(4000))
    base = st.mean(float(before[k]["accept_len"]) for k in ids)
    rows.append({
        "comparison": label, "prompts": len(ids),
        "before_mean": round(base, 3), "after_mean": round(base + st.mean(diffs), 3),
        "mean_change": round(st.mean(diffs), 3), "pct_change": round(100 * st.mean(diffs) / base, 1),
        "ci95_low": round(boot[100], 3), "ci95_high": round(boot[3899], 3),
        "after_higher": sum(d > 0 for d in diffs), "after_lower": sum(d < 0 for d in diffs),
    })
    print(rows[-1])

os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
with open(args.out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)

In [ ]:
%%writefile shift.py
"""How far did a fine-tune move the target? Score each run's outputs with the ORIGINAL model.

top1_agreement: share of output tokens the original model would also have picked greedily
                (about 100% for the original model's own outputs; lower = bigger shift).
mean_nll:       average negative log-likelihood of the output tokens under the original model.

Usage: python shift.py --out results/shift.csv "mild=results/stage2_med_broken.csv" ...
"""
import argparse
import csv
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from common import BASE, chat_ids, read_jsonl

p = argparse.ArgumentParser()
p.add_argument("runs", nargs="+", help="label=bench.csv")
p.add_argument("--prompts", default="prompts/med_eval.jsonl")
p.add_argument("--out", required=True)
p.add_argument("--n", type=int, default=100)
args = p.parse_args()

tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map={"": 0}).eval()
prompts = {ex["id"]: ex["prompt"] for ex in read_jsonl(args.prompts)}

rows = []
for run in args.runs:
    label, path = run.split("=", 1)
    agree = total = 0
    nll = 0.0
    for r in list(csv.DictReader(open(path)))[: args.n]:
        prompt_ids = chat_ids(tok, prompts[r["id"]])
        out_ids = tok(r["ea_text"], add_special_tokens=False).input_ids
        if not out_ids:
            continue
        ids = torch.tensor([prompt_ids + out_ids], device="cuda")
        with torch.no_grad():
            logits = model(input_ids=ids).logits[0, len(prompt_ids) - 1:-1].float()
        target = ids[0, len(prompt_ids):]
        agree += (logits.argmax(-1) == target).sum().item()
        nll += torch.nn.functional.cross_entropy(logits, target, reduction="sum").item()
        total += len(out_ids)
    rows.append({"run": label, "tokens": total, "top1_agreement": round(agree / total, 4),
                 "mean_nll": round(nll / total, 4)})
    print(rows[-1])

os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
with open(args.out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)

In [ ]:
%%writefile train_target_lora.py
"""Stage 2: LoRA fine-tune the target (Qwen3-1.7B) on MedMCQA, then merge and save it.

The merged model is saved in 2 GB shards so it has a model.safetensors.index.json,
which the EAGLE head needs to find the target's embeddings.
"""
import argparse
import csv
import os
import random

import torch
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

from common import BASE, chat_ids, read_jsonl, set_seed

p = argparse.ArgumentParser()
p.add_argument("--data", default="data/med_train.jsonl")
p.add_argument("--adapter_out", required=True)
p.add_argument("--merged_out", required=True)
p.add_argument("--log", required=True)
p.add_argument("--max_examples", type=int, default=1000)
p.add_argument("--epochs", type=int, default=1)
p.add_argument("--rank", type=int, default=16)
p.add_argument("--lr", type=float, default=1e-4)
p.add_argument("--grad_accum", type=int, default=8)
p.add_argument("--max_len", type=int, default=768)
args = p.parse_args()
set_seed()

tok = AutoTokenizer.from_pretrained(BASE)
end_ids = tok("<|im_end|>\n", add_special_tokens=False).input_ids

examples = []
for ex in read_jsonl(args.data)[: args.max_examples]:
    prompt = chat_ids(tok, ex["prompt"])
    answer = tok(ex["answer"], add_special_tokens=False).input_ids + end_ids
    ids = (prompt + answer)[: args.max_len]
    labels = ([-100] * len(prompt) + answer)[: args.max_len]
    examples.append((ids, labels))
schedule = []  # each epoch is a fresh shuffle of the examples
for _ in range(args.epochs):
    order = list(examples)
    random.shuffle(order)
    schedule += order
examples = schedule

if not os.path.exists(os.path.join(args.adapter_out, "adapter_config.json")):
    model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map={"": 0})
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    cfg = LoraConfig(r=args.rank, lora_alpha=2 * args.rank, lora_dropout=0.05, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, cfg)
    for param in model.parameters():  # fp16 base, fp32 adapter weights for stable updates
        if param.requires_grad:
            param.data = param.data.float()
    model.print_trainable_parameters()

    opt = torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=args.lr)
    scaler = torch.amp.GradScaler("cuda")
    model.train()
    os.makedirs(os.path.dirname(args.log) or ".", exist_ok=True)
    log = csv.writer(open(args.log, "w", newline=""))
    log.writerow(["step", "loss"])
    running = []
    for i, (ids, labels) in enumerate(examples):
        ids_t = torch.tensor([ids], device="cuda")
        labels_t = torch.tensor([labels], device="cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(input_ids=ids_t, labels=labels_t).loss / args.grad_accum
        scaler.scale(loss).backward()
        running.append(loss.item() * args.grad_accum)
        if (i + 1) % args.grad_accum == 0 or i == len(examples) - 1:
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()
            step = (i + 1 + args.grad_accum - 1) // args.grad_accum
            avg = sum(running) / len(running)
            log.writerow([step, round(avg, 4)])
            running = []
            if step % 10 == 0 or i == len(examples) - 1:
                print(f"step {step} loss {avg:.4f}")
    model.save_pretrained(args.adapter_out)
    del model, opt
    torch.cuda.empty_cache()
else:
    print("adapter exists, skipping training:", args.adapter_out)

base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map={"": 0})
merged = PeftModel.from_pretrained(base, args.adapter_out).merge_and_unload()
merged.save_pretrained(args.merged_out, max_shard_size="2GB", safe_serialization=True)
tok.save_pretrained(args.merged_out)
assert os.path.exists(os.path.join(args.merged_out, "model.safetensors.index.json")), "missing shard index"
print("merged target saved to", args.merged_out)

In [ ]:
%%writefile gen_head_data.py
"""Stage 3 data: run the LoRA-tuned target (greedy) on in-domain prompts and save prompt+output token ids.

Generation uses EAGLE with the original head only for speed: greedy speculative decoding is
verified by the target, so the text is the target's own greedy output. Resumable.
"""
import argparse
import json
import os

import torch

from common import HEAD, chat_ids, load_ea_model, read_jsonl, set_seed

p = argparse.ArgumentParser()
p.add_argument("--base", required=True, help="merged LoRA-tuned target")
p.add_argument("--prompts", default="prompts/med_headtrain.jsonl")
p.add_argument("--out", required=True)
p.add_argument("--n", type=int, default=600)
p.add_argument("--max_new_tokens", type=int, default=256)
args = p.parse_args()
set_seed()

model = load_ea_model(args.base, HEAD)
tok = model.get_tokenizer()

done = set()
if os.path.exists(args.out):
    done = {json.loads(line)["id"] for line in open(args.out)}
os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
with open(args.out, "a") as f:
    prompts = read_jsonl(args.prompts)[: args.n]
    for i, ex in enumerate(prompts):
        if ex["id"] in done:
            continue
        prompt_ids = chat_ids(tok, ex["prompt"])
        out = model.eagenerate(torch.tensor([prompt_ids], device="cuda"), temperature=0.0,
                               max_new_tokens=args.max_new_tokens)
        f.write(json.dumps({"id": ex["id"], "prompt_len": len(prompt_ids), "input_ids": out[0].tolist()}) + "\n")
        f.flush()
        if (i + 1) % 25 == 0:
            print(f"{i + 1}/{len(prompts)}")
print("saved", args.out)

In [ ]:
%%writefile train_head_lora.py
"""Stage 3: train a LoRA adapter on the EAGLE-3 draft head against the LoRA-tuned target.

The head's base weights stay frozen. For each sequence the target's hidden states (3 taps)
and logits are computed on the fly; the head is trained to match the target's next-token
distribution over the draft vocabulary, with the same loss as EAGLE-3's training code.

Simplification vs. the official EAGLE-3 recipe: only the first drafting step is trained
(teacher-forced on target features). The official recipe also unrolls later draft steps
("training-time test"). Noted in the README.

Writes the adapter, a per-epoch log (loss + top-1 agreement on held-out sequences), and
results/adapter_size.csv comparing adapter size with the full head.
"""
import argparse
import csv
import os
import random

import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download

from common import HEAD, load_ea_model, read_jsonl, set_seed
from head_lora import add_head_lora, save_head_adapter

p = argparse.ArgumentParser()
p.add_argument("--base", required=True, help="merged LoRA-tuned target")
p.add_argument("--data", required=True, help="output of gen_head_data.py")
p.add_argument("--out", required=True, help="adapter folder")
p.add_argument("--log", required=True)
p.add_argument("--size_csv", default=None)
p.add_argument("--max_seqs", type=int, default=100000)
p.add_argument("--rank", type=int, default=16)
p.add_argument("--epochs", type=int, default=2)
p.add_argument("--lr", type=float, default=2e-4)
p.add_argument("--grad_accum", type=int, default=8)
p.add_argument("--val_frac", type=float, default=0.1)
args = p.parse_args()
set_seed()

model = load_ea_model(args.base, HEAD)
head = model.ea_layer
head.float()  # train in fp32; the target stays fp16
head.reset()  # no tree mask: plain causal attention during training
meta = add_head_lora(head, args.rank)
trainable = [q for q in head.parameters() if q.requires_grad]
print(f"trainable adapter params: {sum(q.numel() for q in trainable):,}")

seqs = read_jsonl(args.data)[: args.max_seqs]
random.shuffle(seqs)
n_val = max(1, int(len(seqs) * args.val_frac))
val, train = seqs[:n_val], seqs[n_val:]
print(f"train seqs={len(train)} val seqs={len(val)}")


def loss_and_acc(ex):
    ids = torch.tensor([ex["input_ids"]], device="cuda")
    with torch.no_grad():
        out = model.base_model.model(input_ids=ids)
        logits = model.base_model.lm_head(out[0])
        feats = torch.cat(out["hidden_states"], dim=-1)
    # Same alignment as EAGLE drafting: target features at position t + token t+1
    # -> predict the target's distribution for token t+2.
    hs_in = feats[:, :-1].float()
    ids_in = ids[:, 1:]
    target = logits[:, 1:].float()
    hidden = head(hs_in, input_ids=ids_in)
    draft_logp = F.log_softmax(head.lm_head(head.norm(hidden)).float(), dim=-1)
    with torch.no_grad():
        t2d = head.t2d
        target_p = F.softmax(target[..., t2d], dim=-1)
        mask = t2d[target.argmax(-1)].float()  # target's top token is in the draft vocab
        start = max(ex["prompt_len"] - 2, 0)  # only score the response tokens
        mask[:, :start] = 0
    denom = mask.sum().clamp(min=1)
    loss = -(mask * (target_p * draft_logp).sum(-1)).sum() / denom
    acc = ((draft_logp.argmax(-1) == target_p.argmax(-1)).float() * mask).sum() / denom
    return loss, acc.item()


def evaluate():
    head.eval()
    with torch.no_grad():
        res = [loss_and_acc(ex) for ex in val]
    head.train()
    return sum(r[0].item() for r in res) / len(res), sum(r[1] for r in res) / len(res)


opt = torch.optim.AdamW(trainable, lr=args.lr)
os.makedirs(os.path.dirname(args.log) or ".", exist_ok=True)
log = csv.writer(open(args.log, "w", newline=""))
log.writerow(["epoch", "train_loss", "val_loss", "val_top1_agreement"])
vl, va = evaluate()
log.writerow([0, "", round(vl, 4), round(va, 4)])
print(f"before training: val_loss={vl:.4f} val_top1={va:.4f}")

head.train()
for epoch in range(1, args.epochs + 1):
    random.shuffle(train)
    total = 0.0
    for i, ex in enumerate(train):
        loss, _ = loss_and_acc(ex)
        (loss / args.grad_accum).backward()
        total += loss.item()
        if (i + 1) % args.grad_accum == 0 or i == len(train) - 1:
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            opt.step()
            opt.zero_grad()
    vl, va = evaluate()
    log.writerow([epoch, round(total / len(train), 4), round(vl, 4), round(va, 4)])
    print(f"epoch {epoch}: train_loss={total / len(train):.4f} val_loss={vl:.4f} val_top1={va:.4f}")

sd = save_head_adapter(head, meta, args.out)
print("adapter saved to", args.out)

if args.size_csv:
    head_file = hf_hub_download(HEAD, "pytorch_model.bin")
    head_sd = torch.load(head_file, map_location="cpu")
    head_params = sum(v.numel() for k, v in head_sd.items() if k not in ("d2t", "t2d"))
    adapter_file = os.path.join(args.out, "head_lora.pt")
    row = {
        "adapter_params": sum(v.numel() for v in sd.values()),
        "adapter_file_mb": round(os.path.getsize(adapter_file) / 1e6, 2),
        "head_params": head_params,
        "head_file_mb": round(os.path.getsize(head_file) / 1e6, 2),
    }
    row["param_ratio_pct"] = round(100 * row["adapter_params"] / row["head_params"], 2)
    os.makedirs(os.path.dirname(args.size_csv) or ".", exist_ok=True)
    with open(args.size_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row))
        w.writeheader()
        w.writerow(row)
    print(row)

In [ ]:
%%writefile plot.py
"""Stage 4 chart: speedup over plain decoding for the three setups, on medical and general prompts.

Reads two summary CSVs written by summarize.py, each from a single Kaggle session
(so speeds within a group are directly comparable).
"""
import argparse
import csv

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

p = argparse.ArgumentParser()
p.add_argument("--medical", default="results/summary.csv")
p.add_argument("--general", default="results/summary_stage3b.csv")
p.add_argument("--out", default="results/chart.png")
args = p.parse_args()

SERIES = [  # fixed order and colours (validated categorical palette, slots 1-3)
    ("Original model, original head", "#2a78d6"),
    ("Strong LoRA, original head", "#eb6834"),
    ("Strong LoRA, head + LoRA adapter", "#1baf7a"),
]
groups = []
for label, path in [("Medical prompts (MedMCQA)", args.medical), ("General prompts (Alpaca)", args.general)]:
    rows = list(csv.DictReader(open(path)))
    assert len(rows) == 3, f"{path}: expected the 3 runs in SERIES order"
    groups.append((label, [float(r["speedup"]) for r in rows], [float(r["accept_len"]) for r in rows]))

fig, ax = plt.subplots(figsize=(9, 4.8))
width, gap = 0.26, 0.02
for g, (glabel, speedups, accepts) in enumerate(groups):
    for i, ((name, color), s, a) in enumerate(zip(SERIES, speedups, accepts)):
        x = g + (i - 1) * (width + gap)
        ax.bar(x, s, width, color=color, label=name if g == 0 else None)
        ax.text(x, s + 0.03, f"{s:.2f}x", ha="center", va="bottom", fontsize=10, color="#0b0b0b")
        ax.text(x, 0.06, f"{a:.2f}\ntok/step", ha="center", va="bottom", fontsize=8, color="#0b0b0b")
ax.set_xticks(range(len(groups)), [g[0] for g in groups], fontsize=10, color="#52514e")
ax.set_ylabel("Speedup over plain decoding", fontsize=10, color="#52514e")
ax.set_ylim(0, 2.4)
ax.axhline(1.0, color="#52514e", linewidth=1, linestyle="--")
ax.grid(axis="y", color="#e8e7e3", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", colors="#52514e", length=0)
ax.tick_params(axis="x", length=0)
ax.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
ax.set_title("A 1.7 MB adapter on the EAGLE-3 head restores the speedup lost to a LoRA fine-tune",
             fontsize=11, loc="left", color="#0b0b0b")
fig.text(0.01, 0.005, "Qwen3-1.7B + AngelSlim EAGLE-3 head, Kaggle T4, greedy, max 256 new tokens, 100 prompts per bar",
         fontsize=8, color="#52514e")
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")
fig.tight_layout(rect=(0, 0.02, 1, 1))
fig.savefig(args.out, dpi=160)
print("saved", args.out)

## Stage 1 - Baseline
Original Qwen3-1.7B + original EAGLE-3 head on 100 general (Alpaca) prompts.

In [ ]:
!python make_prompts.py

In [ ]:
# Smoke test: head keys match, and EAGLE output equals plain greedy output
!python check_head.py

In [ ]:
# Small run first: 5 prompts, 128 tokens
!python bench.py --prompts prompts/general_eval.jsonl --n 5 --max_new_tokens 128 --out results/smoke/stage1.csv
!python summarize.py "smoke=results/smoke/stage1.csv"

In [ ]:
# Real run: 100 prompts, 256 tokens
!python bench.py --prompts prompts/general_eval.jsonl --out results/stage1_baseline.csv
!python summarize.py "baseline (general)=results/stage1_baseline.csv" --out results/summary_stage1.csv

In [ ]:
!cd /kaggle/working && zip -qr drafter-lora-results.zip drafter-lora/results drafter-lora/prompts && echo zipped

**Stop here.** Send Claude the outputs above before starting Stage 2.

## Stage 2 - Break it
LoRA fine-tune the target on MedMCQA, then run the *original* head against it on 100 held-out
medical prompts. The original target is also run on the same medical prompts, so the drop isn't
confused with a change of domain.

In [ ]:
# Rebuild the training data with the cleaned-explanation filter. The eval prompts must not change:
# compare the two checksums printed below.
!md5sum prompts/med_eval.jsonl prompts/general_eval.jsonl 2>/dev/null
!python make_prompts.py
!md5sum prompts/med_eval.jsonl prompts/general_eval.jsonl
!head -c 600 data/med_train.jsonl

In [ ]:
# Small run first: 40 examples, then a 3-prompt benchmark
!python train_target_lora.py --max_examples 40 --grad_accum 4 --adapter_out /kaggle/working/smoke/target_lora --merged_out /kaggle/working/smoke/target_merged --log results/smoke/target_lora_loss.csv
!python bench.py --base /kaggle/working/smoke/target_merged --prompts prompts/med_eval.jsonl --n 3 --max_new_tokens 64 --out results/smoke/stage2.csv
!python summarize.py "smoke=results/smoke/stage2.csv"
!rm -rf /kaggle/working/smoke/target_lora /kaggle/working/smoke/target_merged

In [ ]:
# Real run: 1000 examples, 1 epoch
!python train_target_lora.py --adapter_out /kaggle/working/target_lora --merged_out /kaggle/working/target_merged --log results/target_lora_loss.csv

In [ ]:
!python bench.py --prompts prompts/med_eval.jsonl --out results/stage2_med_original.csv
!python bench.py --base /kaggle/working/target_merged --prompts prompts/med_eval.jsonl --out results/stage2_med_broken.csv
!python summarize.py "baseline (general)=results/stage1_baseline.csv" "original target (medical)=results/stage2_med_original.csv" "LoRA target, original head (medical)=results/stage2_med_broken.csv" --out results/summary_stage2.csv

In [ ]:
!cd /kaggle/working && zip -qr drafter-lora-results.zip drafter-lora/results drafter-lora/prompts target_lora && echo zipped

**Stop here.** Send Claude the table. If accepted tokens per step barely dropped, that is a real
finding and Stage 3 should change.

## Stage 2b - How hard do you have to fine-tune before the head falls behind?
The mild fine-tune (1,000 examples, 1 epoch, rank 16) did not lower acceptance. Two stronger
fine-tunes on the same medical data, each benchmarked on the same 100 eval prompts.
`shift.py` measures how far each fine-tune moved the model: the share of its output tokens the
original model would also have picked (the original model's own outputs should be near 100%).

In [ ]:
# 2,000 training examples (eval prompts must be unchanged: compare the checksum with Stage 2's)
!python make_prompts.py --n_target_train 2000
!md5sum prompts/med_eval.jsonl prompts/general_eval.jsonl

In [ ]:
# Small run first: 16 examples x 2 epochs, 3-prompt benchmark, shift on 3 outputs
!python train_target_lora.py --max_examples 16 --epochs 2 --rank 64 --grad_accum 4 --adapter_out /kaggle/working/smoke/l2_lora --merged_out /kaggle/working/smoke/l2_merged --log results/smoke/l2_loss.csv
!python bench.py --base /kaggle/working/smoke/l2_merged --prompts prompts/med_eval.jsonl --n 3 --max_new_tokens 64 --out results/smoke/stage2b.csv
!python shift.py --n 3 --out results/smoke/shift.csv "original=results/stage2_med_original.csv" "smoke=results/smoke/stage2b.csv"
!rm -rf /kaggle/working/smoke

In [ ]:
# Medium: 2,000 examples x 3 epochs, rank 64, lr 1e-4
!python train_target_lora.py --max_examples 2000 --epochs 3 --rank 64 --lr 1e-4 --adapter_out /kaggle/working/target_lora_medium --merged_out /kaggle/working/target_merged_medium --log results/target_lora_loss_medium.csv
!python bench.py --base /kaggle/working/target_merged_medium --prompts prompts/med_eval.jsonl --out results/stage2b_med_medium.csv
!rm -rf /kaggle/working/target_merged_medium

In [ ]:
# Strong: same, lr 3e-4
!python train_target_lora.py --max_examples 2000 --epochs 3 --rank 64 --lr 3e-4 --adapter_out /kaggle/working/target_lora_strong --merged_out /kaggle/working/target_merged_strong --log results/target_lora_loss_strong.csv
!python bench.py --base /kaggle/working/target_merged_strong --prompts prompts/med_eval.jsonl --out results/stage2b_med_strong.csv
!rm -rf /kaggle/working/target_merged_strong

In [ ]:
!python shift.py --out results/shift.csv "original=results/stage2_med_original.csv" "mild=results/stage2_med_broken.csv" "medium=results/stage2b_med_medium.csv" "strong=results/stage2b_med_strong.csv"
!python summarize.py "original target (medical)=results/stage2_med_original.csv" "mild LoRA=results/stage2_med_broken.csv" "medium LoRA=results/stage2b_med_medium.csv" "strong LoRA=results/stage2b_med_strong.csv" --out results/summary_stage2b.csv

In [ ]:
!cd /kaggle/working && zip -qr drafter-lora-results.zip drafter-lora/results drafter-lora/prompts target_lora_medium target_lora_strong && echo zipped

**Stop here.** Send Claude the two tables.

## Stage 3 - Fix it
Target: the **strong** fine-tune from Stage 2b (its LoRA adapter is re-merged here, no retraining).
Generate data with it, train a LoRA adapter on the draft head (base weights frozen), and benchmark
on the same 100 medical prompts. The original model and the strong model with the original head are
re-run in this same session, so all speeds in the final table come from one machine.

In [ ]:
# Head-training prompts must not overlap the strong model's 2,000 training examples:
# regenerate them (rows 2000-2600 of the shuffled cleaned train split). Eval checksum must match Stage 2.
!python make_prompts.py --n_target_train 2000
!md5sum prompts/med_eval.jsonl prompts/general_eval.jsonl
!python -c "import json; a={json.loads(l)['id'] for l in open('data/med_train.jsonl')}; b={json.loads(l)['id'] for l in open('prompts/med_headtrain.jsonl')}; print('train', len(a), 'headtrain', len(b), 'overlap', len(a & b))"
# Re-merge the strong target from its saved adapter (training is skipped because the adapter exists)
!python train_target_lora.py --max_examples 2000 --epochs 3 --rank 64 --lr 3e-4 --adapter_out /kaggle/working/target_lora_strong --merged_out /kaggle/working/target_merged_strong --log results/smoke/remerge_log.csv

In [ ]:
# Small run first: 10 sequences, 1 epoch, 3-prompt benchmark
!python gen_head_data.py --base /kaggle/working/target_merged_strong --n 10 --out /kaggle/working/smoke/head_data.jsonl
!python train_head_lora.py --base /kaggle/working/target_merged_strong --data /kaggle/working/smoke/head_data.jsonl --epochs 1 --grad_accum 2 --out /kaggle/working/smoke/head_lora --log results/smoke/head_lora_log.csv
!python bench.py --base /kaggle/working/target_merged_strong --adapter /kaggle/working/smoke/head_lora --prompts prompts/med_eval.jsonl --n 3 --max_new_tokens 64 --out results/smoke/stage3.csv
!python summarize.py "smoke=results/smoke/stage3.csv"

In [ ]:
# Real run: 600 sequences, 2 epochs
!python gen_head_data.py --base /kaggle/working/target_merged_strong --out data/head_data.jsonl
!python train_head_lora.py --base /kaggle/working/target_merged_strong --data data/head_data.jsonl --out adapters/head_lora --log results/head_lora_log.csv --size_csv results/adapter_size.csv
!cat results/head_lora_log.csv

In [ ]:
# All three medical runs in this one session
!python bench.py --prompts prompts/med_eval.jsonl --out results/stage3_med_original.csv
!python bench.py --base /kaggle/working/target_merged_strong --prompts prompts/med_eval.jsonl --out results/stage3_med_strong_orighead.csv
!python bench.py --base /kaggle/working/target_merged_strong --adapter adapters/head_lora --prompts prompts/med_eval.jsonl --out results/stage3_med_strong_fixed.csv
!python summarize.py "original model=results/stage3_med_original.csv" "strong LoRA, original head=results/stage3_med_strong_orighead.csv" "strong LoRA, head + adapter=results/stage3_med_strong_fixed.csv" --out results/summary.csv
!cat results/adapter_size.csv

In [ ]:
!cd /kaggle/working && zip -qr drafter-lora-results.zip drafter-lora/results drafter-lora/prompts drafter-lora/adapters && echo zipped

**Stop here.** Send Claude the table, the training log and the adapter size.

## Stage 3b - Does the head adapter hurt outside the medical domain?
The adapter was trained on medical answers only. Re-run the 100 general (Alpaca) prompts from
Stage 1: original model, strong fine-tune with the original head, strong fine-tune with head + adapter.

In [ ]:
!md5sum prompts/general_eval.jsonl
!ls adapters/head_lora
!python train_target_lora.py --max_examples 2000 --epochs 3 --rank 64 --lr 3e-4 --adapter_out /kaggle/working/target_lora_strong --merged_out /kaggle/working/target_merged_strong --log results/smoke/remerge_log.csv

In [ ]:
!python bench.py --prompts prompts/general_eval.jsonl --out results/stage3b_gen_original.csv
!python bench.py --base /kaggle/working/target_merged_strong --prompts prompts/general_eval.jsonl --out results/stage3b_gen_strong_orighead.csv
!python bench.py --base /kaggle/working/target_merged_strong --adapter adapters/head_lora --prompts prompts/general_eval.jsonl --out results/stage3b_gen_strong_fixed.csv
!python summarize.py "original model (general)=results/stage3b_gen_original.csv" "strong LoRA, original head (general)=results/stage3b_gen_strong_orighead.csv" "strong LoRA, head + adapter (general)=results/stage3b_gen_strong_fixed.csv" --out results/summary_stage3b.csv

**Stop here.** Send Claude the table.

## Stage 4 - Paired statistics and chart (no GPU needed; also runs locally on the results CSVs)

In [ ]:
!python paired.py --out results/paired_stats.csv "medical: adapter vs no adapter=results/stage3_med_strong_orighead.csv,results/stage3_med_strong_fixed.csv" "general: adapter vs no adapter=results/stage3b_gen_strong_orighead.csv,results/stage3b_gen_strong_fixed.csv"
!python plot.py --medical results/summary.csv --general results/summary_stage3b.csv --out results/chart.png
from IPython.display import Image
Image("results/chart.png")